# C9-dimensionality-reduction — Practice p09 — Solution

In [1]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["sparrow", "pigeon", "crow", "robin", "owl", "swan",
         "apple", "mango", "plum", "cherry", "peach", "grape",
         "hammer", "wrench", "chisel", "pliers", "saw", "drill",
         "jacket", "scarf", "sweater", "collar", "sleeve", "cloak"]
BUDGETS = (0.20, 0.10, 0.02)
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
_, s, _ = np.linalg.svd(W, full_matrices=False)
fro2 = float((s * s).sum())
rel_err2 = (fro2 - np.cumsum(s**2)) / fro2
r_star = {}
for budget in BUDGETS:
    rank = int(np.argmax(rel_err2 <= budget) + 1)
    r_star[budget] = rank
    assert rel_err2[rank - 1] <= budget
    assert rank == 1 or rel_err2[rank - 2] > budget

print("budget ranks:", r_star)
print("chosen errors:", rel_err2[10], rel_err2[14], rel_err2[20])

budget ranks: {0.2: 11, 0.1: 15, 0.02: 21}
chosen errors: 0.17625740961857086 0.08765754401752712 0.018266446094640305


The spectrum alone selects ranks $11$, $15$, and $21$ for budgets $0.20$, $0.10$, and $0.02$.  Each selected entry meets its budget, while the immediately preceding rank fails it.

### Answer check

In [2]:
assert W.shape == (24, 100)
assert W.dtype == np.float64
assert rel_err2.shape == (24,)
assert r_star == {0.20: 11, 0.10: 15, 0.02: 21}
assert np.isclose(rel_err2[10], 0.176257410, atol=1e-9, rtol=0)
assert np.isclose(rel_err2[14], 0.0876575440, atol=1e-10, rtol=0)
assert np.isclose(rel_err2[20], 0.0182664461, atol=1e-10, rtol=0)